# Baseline CNN

Small CNN trained from scratch, like the cats vs dogs example from class: conv + max pooling blocks, data augmentation and dropout.

Only train and validation are used here. The test set is used once, at the end of the project.

Runtime > Change runtime type > T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the zip from Drive and unzip it on the Colab disk (1-2 min)
!cp /content/drive/MyDrive/plantnet_dlnn26/plantnet20_v1.zip /content/
!unzip -q -o /content/plantnet20_v1.zip -d /content/plantnet20

In [ ]:
# Code shared by the group (src/)
!git clone -q https://github.com/aphils12/plant-identifier_dlnn26.git /content/repo
import sys
sys.path.append('/content/repo')

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from src.callbacks import get_callbacks
from src.experiments import log_experiment

drive_dir = '/content/drive/MyDrive/plantnet_dlnn26'
data_dir = '/content/plantnet20'
models_dir = drive_dir + '/models'
os.makedirs(models_dir, exist_ok=True)

# 20 species, in pairs of the same genus (same order as the README)
species_ids = [4, 3, 112, 225, 84, 171, 200, 202, 190, 15, 11, 406, 47, 46, 60, 186, 35, 243, 310, 366]

author = ''      # your name, for the experiment log
model_name = 'baseline_cnn'

img_size = 180   # same as the cats vs dogs example
batch_size = 32
seed = 42
keras.utils.set_random_seed(seed)

In [ ]:
sel = pd.read_csv(drive_dir + '/selection.csv')
sel['file'] = data_dir + '/' + sel['path']

# label = position of the species in species_ids (0 to 19)
sel['label'] = sel['species_id'].map({s: i for i, s in enumerate(species_ids)})
names = sel.drop_duplicates('species_id').set_index('species_id')['species']
class_names = [names[s] for s in species_ids]

train_df = sel[sel['split'] == 'train']
val_df = sel[sel['split'] == 'val']
print(len(train_df), len(val_df))

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (img_size, img_size))
    return img, label

def make_dataset(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df['file'].values, df['label'].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=seed)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)

for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)   # (32, 180, 180, 3) (32,)

## Model

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
])

inputs = keras.Input(shape=(img_size, img_size, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1. / 255)(x)
x = layers.Conv2D(32, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(256, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(256, 3, activation='relu')(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(20, activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer='rmsprop',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = get_callbacks(models_dir + '/' + model_name + '.keras')

start = time.time()
history = model.fit(train_ds,
                    epochs=50,
                    validation_data=val_ds,
                    callbacks=callbacks)
train_time = time.time() - start

pd.DataFrame(history.history).to_csv(models_dir + '/' + model_name + '_history.csv', index=False)

## Training curves

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.legend()
plt.show()

plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.legend()
plt.show()

## Results on validation

Macro-F1 is the main metric because val and test are not balanced.

In [ ]:
probs = model.predict(val_ds)
y_pred = probs.argmax(axis=1)
y_true = val_df['label'].values

print('accuracy:', round((y_pred == y_true).mean(), 3))
print('macro F1:', round(f1_score(y_true, y_pred, average='macro'), 3))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(20))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, xticks_rotation=90, colorbar=False)
plt.title('Confusion matrix (validation)')
plt.show()

In [ ]:
# How many errors are with the other species of the same genus?
# pairs are next to each other in species_ids, so the pair of label i is i ^ 1
wrong = y_pred != y_true
same_genus = wrong & (y_pred == (y_true ^ 1))
print('errors:', wrong.sum())
print('errors with the other species of the pair:', same_genus.sum(), '(', round(same_genus.sum() / wrong.sum() * 100, 1), '% )')

# F1 per species
per_class = pd.Series(f1_score(y_true, y_pred, labels=range(20), average=None), index=class_names).round(3)
per_class.sort_values()

In [ ]:
# Flower vs leaf
val_res = val_df.assign(correct=(y_pred == y_true))
val_res.groupby('organ')['correct'].agg(['mean', 'count']).round(3)

## Experiment log

One row per run, in the shared Drive folder.

In [ ]:
log_experiment(drive_dir + '/experiments.csv',
               model=model_name,
               author=author,
               img_size=img_size,
               batch_size=batch_size,
               augmentation='flip, rotation 0.1, zoom 0.2',
               dropout=0.5,
               optimizer='rmsprop',
               seed=seed,
               epochs=len(history.history['loss']),
               params=model.count_params(),
               train_min=round(train_time / 60, 1),
               val_accuracy=round((y_pred == y_true).mean(), 4),
               val_macro_f1=round(f1_score(y_true, y_pred, average='macro'), 4),
               notes='')